In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
df = pd.read_csv('../../data/raw/bank_sim.csv')

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().any()

In [ ]:
df[df.duplicated()]

### Sort columns into their data type

In [ ]:
cat_cols, cont_cols = [], []

for col in df.columns:
    if df[col].dtype == "str":
        cat_cols.append(col)
    else:
        cont_cols.append(col)

print("Categorical Columns:", cat_cols)
print("Continuous Columns:", cont_cols)

### Check for columns with no variability
If they have no variance they have no use to a model due to thier features being constant

In [ ]:
cols_to_drop = []
for col in cat_cols:
   unum = df[col].nunique()

   print(f"Unique numbers of {col}s:", unum)

   if unum == 1:
      cols_to_drop.append(col)

print("\nDropping columns due to constant values:", cols_to_drop)

for col in cols_to_drop:
   cat_cols.remove(col)

df.drop(columns=cols_to_drop, inplace=True)

### Extract Customer-based Features (All local features)
These include: 

- "prev_step" - The last time they did a transaction (-1 if no previous transaction exists)
- "time_since_last_transaction" --
- "transaction_count" - Number of transactions

- "amount_sum" - Sum of all previous transactions
- "avg_amount" - Avergae of all past transactions
- "std_amount" - Standard deviation of past transaction amounts
- "amount_zscore" - How many standard deviation this transaction is from past behaviour

- "avg_amount_ratio" - Ratio of how this given transaction compares to customers average
- "log_amount_ratio" - Log transformation to compress extreme values

- "merchant_count" - Number of times a customer has shopped with this merchant
- "category_count" - Number of times a customer has shopped this category
- "prev_merchant_step" - Last time they shopped with this merchant (-1 if they never have)
- "time_since_last_merchant_transaction" --

In [ ]:
# Just checking if the rolling features are working as expected (NEED TO REVERT THIS SORTING BACK TO ORIGINAL ORDER LATER)
df = df.sort_values(["customer", "step"]).copy()

cust_groups = df.groupby("customer")

# Shift gets the previous step for each customer
df["prev_step"] = cust_groups["step"].shift(1)
df["time_since_last_transaction"] = df["step"] - df["prev_step"]

df["time_since_last_transaction"] = df["time_since_last_transaction"].fillna(-1)
df["prev_step"] = df["prev_step"].fillna(-1)

df["transaction_count"] = cust_groups.cumcount()

df["amount_sum"] = cust_groups["amount"].cumsum() - df["amount"] 
df["amount_sum"] = df["amount_sum"].fillna(0)

df["avg_amount"] = df["amount_sum"] / df["transaction_count"]
df["avg_amount"] = df["avg_amount"].fillna(df["amount"])

# Expanding std gives the std of all previous transactions, shift(1) makes it so that the current transaction is not included in the std calculation
df["std_amount"] = cust_groups["amount"].expanding().std().reset_index(level=0, drop=True).shift(1)
df["std_amount"] = df["std_amount"].fillna(0)

df["avg_amount_ratio"] = df["amount"] / df["avg_amount"]
df["log_amount_ratio"] = np.log1p(df["avg_amount_ratio"])

df["amount_zscore"] = (df["amount"] - df["avg_amount"]) / df["std_amount"]
df["amount_zscore"] = df["amount_zscore"].fillna(0)
df["amount_zscore"] = df["amount_zscore"].replace([np.inf, -np.inf], 0)

df["merchant_count"] = df.groupby(["customer", "merchant"]).cumcount()
df["category_count"] = df.groupby(["customer", "category"]).cumcount()

df["prev_merchant_step"] = df.groupby(["customer", "merchant"])["step"].shift(1)
df["time_since_last_merchant_transaction"] = df["step"] - df["prev_merchant_step"]

df["time_since_last_merchant_transaction"] = df["time_since_last_merchant_transaction"].fillna(-1)
df["prev_merchant_step"] = df["prev_merchant_step"].fillna(-1)

df

In [ ]:
# Sort back to original dataset
df = df.sort_index()
df

### Extract Merchant-based features (ALl local features)
All of these are the same as customer based features execept for:
- "merchant_fraud_count" - How many times has fraud occured for this merchant
- "merchant_fraud_rate" - Rate of fraud up to this transaction

In [135]:
df = df.sort_values(["merchant", "step"]).copy()
merchant_groups = df.groupby("merchant")

df["merchant_transaction_count"] = merchant_groups.cumcount()

df["merchant_past_amount_sum"] = merchant_groups["amount"].cumsum() - df["amount"]
df["merchant_past_amount_sum"] = df["merchant_past_amount_sum"].fillna(0)

df["merchant_avg_amount"] = df["merchant_past_amount_sum"] / df["merchant_transaction_count"]
df["merchant_avg_amount"] = df["merchant_avg_amount"].fillna(df["amount"])

df["merchant_std_amount"] = merchant_groups["amount"].expanding().std().reset_index(level=0, drop=True).shift(1)
df["merchant_std_amount"] = df["merchant_std_amount"].fillna(0)

df["merchant_avg_amount_ratio"] = df["amount"] / df["merchant_avg_amount"]
df["merchant_log_amount_ratio"] = np.log1p(df["merchant_avg_amount_ratio"])

df["merchant_amount_zscore"] = (df["amount"] - df["merchant_avg_amount"]) / df["merchant_std_amount"]
df["merchant_amount_zscore"] = df["merchant_amount_zscore"].fillna(0)
df["merchant_amount_zscore"] = df["merchant_amount_zscore"].replace([np.inf, -np.inf], 0)

df["merchant_fraud_count"] = merchant_groups["fraud"].cumsum() - df["fraud"]
df["merchant_fraud_rate"] = df["merchant_fraud_count"] / df["merchant_transaction_count"]
df["merchant_fraud_rate"] = df["merchant_fraud_rate"].fillna(0)

df["merchant_prev_step"] = merchant_groups["step"].shift(1)
df["time_since_last_merchant_transaction"] = df["step"] - df["merchant_prev_step"]

df["time_since_last_merchant_transaction"] = df["time_since_last_merchant_transaction"].fillna(-1)
df["merchant_prev_step"] = df["merchant_prev_step"].fillna(-1)

df


,step,customer,merchant,category,amount,fraud,prev_step,time_since_last_transaction,transaction_count,past_amount_sum,...,age_'2',age_'3',age_'4',age_'5',age_'6',age_'U',gender_'F',gender_'M',gender_'U',merchant_fraud_count
41,0,'C1635613216','M1053599405','es_health',105.59,0,-1.0,-1.0,0,0.00,...,0,0,1,0,0,0,1,0,0,0
78,0,'C118437987','M1053599405','es_health',159.92,0,-1.0,-1.0,0,0.00,...,1,0,0,0,0,0,0,1,0,0
130,0,'C650108285','M1053599405','es_health',11.83,0,-1.0,-1.0,0,0.00,...,0,0,1,0,0,0,1,0,0,0
160,0,'C1463833315','M1053599405','es_health',7.94,0,-1.0,-1.0,0,0.00,...,0,0,0,0,0,0,0,1,0,0
550,0,'C1810630647','M1053599405','es_health',105.54,0,-1.0,-1.0,0,0.00,...,1,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
591304,179,'C1647495093','M980657600','es_sportsandtoys',396.66,1,178.0,1.0,165,35086.88,...,0,1,0,0,0,0,0,1,0,1467
591583,179,'C910454738','M980657600','es_sportsandtoys',203.18,1,177.0,2.0,91,37592.33,...,0,0,1,0,0,0,1,0,0,1468
591584,179,'C2015444413','M980657600','es_sportsandtoys',559.31,1,178.0,1.0,35,2097.97,...,1,0,0,0,0,0,1,0,0,1469
591897,179,'C881073190','M980657600','es_sportsandtoys',224.20,1,177.0,2.0,152,6158.64,...,1,0,0,0,0,0,1,0,0,1470


In [ ]:
# Sort back to original dataset
df = df.sort_index()
df

### Extract Global Features
Features are the same as both customer and merchant just no longer localised features.

In [ ]:
df["global_transaction_count"] = df.index

df["global_amount_sum"] = df["amount"].cumsum() - df["amount"]
df["global_amount_sum"] = df["global_amount_sum"].fillna(0)

df["global_avg_amount"] = df["global_amount_sum"] / df["global_transaction_count"]
df["global_avg_amount"] = df["global_avg_amount"].fillna(df["amount"])

df["global_amount_ratio"] = df["amount"] / df["global_avg_amount"]
df["global_log_amount_ratio"] = np.log1p(df["global_amount_ratio"])

df["global_std_amount"] = df["amount"].expanding().std().shift(1).reset_index(drop=True)
df["global_std_amount"] = df["global_std_amount"].fillna(0)

df["global_z_score"] = (df["amount"] - df["global_avg_amount"]) / df["global_std_amount"]
df["global_z_score"] = df["global_z_score"].fillna(0)
df["global_z_score"] = df["global_z_score"].replace([np.inf, -np.inf], 0)

df

In [ ]:
df.describe()

In [ ]:
df = pd.get_dummies(df, columns=["age", "gender"], drop_first=True)

# Just incase it writes back as string when reading the dataset
bool_cols = df.select_dtypes(include=["bool"]).columns
df[bool_cols] = df[bool_cols].astype(int)

df

In [ ]:
os.makedirs("../../data/processed", exist_ok=True)
df.to_csv("../../data/processed/processed_data.csv", index=False)

In [ ]:
split_step = df["step"].quantile(0.7)
val_step = df["step"].quantile(0.85)

train_df = df[df["step"] <= split_step]
val_df = df[(df["step"] > split_step) & (df["step"] <= val_step)]
test_df = df[df["step"] > val_step]

In [ ]:
os.makedirs("../../data/processed/tree", exist_ok=True)
os.makedirs("../../data/processed/nn", exist_ok=True)

tree_train_df = test_df.copy()
tree_train_df.drop(columns=["customer", "merchant", "category"], inplace=True)
tree_train_df.to_csv("../../data/processed/tree/train_data.csv", index=False)


tree_val_df = val_df.copy()
tree_val_df.drop(columns=["customer", "merchant", "category"], inplace=True)
tree_val_df.to_csv("../../data/processed/tree/val_data.csv", index=False)

tree_test_df = test_df.copy()
tree_test_df.drop(columns=["customer", "merchant", "category"], inplace=True)
tree_test_df.to_csv("../../data/processed/tree/test_data.csv", index=False)

nn_train_df = train_df.copy()
nn_train_df = nn_train_df[nn_train_df["fraud"] == 0]
nn_train_df.to_csv("../../data/processed/nn/train_data.csv", index=False)

nn_val_df = val_df.copy()
nn_val_df.to_csv("../../data/processed/nn/val_data.csv", index=False)

nn_test_df = test_df.copy()
nn_test_df.to_csv("../../data/processed/nn/test_data.csv", index=False)
